# Open AI മോഡലുകൾ ഫൈൻ ട്യൂൺ ചെയ്യൽ

ഈ നോട്ട്‌ബുക്ക് Open AI ന്റെ [ഫൈൻ ട്യൂണിംഗ്](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst) ഡോക്യുമെന്റേഷനിൽ നൽകിയ ഇതുവരെ പ്രയോഗമുള്ള മാർഗ്ഗനിർദ്ദേശങ്ങളുടെ അടിസ്ഥാനത്തിലാണ്.

> **ഉദ്ധരണം:** ഈ നോട്ട്‌ബുക്കിലെ ഉദാഹരണ ഔട്ട്പുട്ട് `gpt-3.5-turbo` മോഡലിനെതിരെ സൃഷ്ടിക്കപ്പെട്ടതാണ്, ഇത് ഇപ്പോൾ Azure OpenAI യിലും Microsoft Foundryവിലും ഇഫിൻറെൻസ്, ഫൈൻ-ട്യൂണിംഗ് ഇരുപ്പും ഇൻഹിബിറ്റ് ചെയ്തിട്ടുള്ളതും OpenAI വൾക്കാതെ നിർവഹിക്കുന്നത്. താഴെ കൊടുത്തിട്ടുള്ള വാക്ക്രോ അഥവാ സിദ്ധാന്തങ്ങൾ ഇപ്പോൾ കാര്യക്ഷമമാണ്, എന്നാൽ നിങ്ങൾ ഇന്ന് പുതുതായി ഫൈൻ-ട്യൂണിംഗ് ജോബ് ആരംഭിക്കുകയാണെങ്കിൽ പ്രാബല്യത്തിലുള്ള മോഡലുകൾ ലക്ഷ്യമിടുക - ഉദാഹരണത്തിന് `gpt-4o-mini` അല്ലെങ്കിൽ `gpt-4.1-mini`. [ഫൈൻ-ട്യൂണിംഗ് മോഡലുകളുടെ പട്ടിക](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models) നിലവിലെ ഫൈൻ-ട്യൂണിംഗ് കഴിയുന്ന മോഡലുകൾക്കായി നോക്കുക.

ഫൈൻ-ട്യൂണിംഗ് നിങ്ങളുടെ അപേക്ഷയ്ക്ക് അനുയോജ്യമായ അധിക ഡാറ്റയും സന്ദർഭവും ഉപയോഗിച്ച് ഫൗണ്ടേഷൻ മോഡലുകളുടെ പ്രവർത്തനം മെച്ചപ്പെടുത്തുന്നതാണ്. ചുരുക്കത്തിൽ, `_few shot learning_`ഉം `_retrieval augmented generation_`ഉം പോലെയുള്ള പ്രൊംപ്റ്റ് എഞ്ചിനീയറിംഗ് സാങ്കേതികതകൾ ഉപയോഗിച്ച് ഡിഫോള്റ് പ്രോംപ്റ്റിനെ അനുയോജ്യമായ ഡാറ്റയാൽ മെച്ചപ്പെടുത്താം. എന്നാല്‍, ഈ സമീപനങ്ങൾ ലക്ഷ്യമിട്ട ഫൗണ്ടേഷൻ മോഡലിന്റെ പരമാവധി ടോക്കൺ വിൻഡോ സൈസ് കൊണ്ട് പരിമിതപ്പെടുത്തിയതാണ്.

ഫൈൻ-ട്യൂണിംഗിലൂടെ, ആവശ്യമായ ഡാറ്റ ഉപയോഗിച്ച് മോഡലിനെ തത്വത്തിൽ വീണ്ടും പരിശീലിപ്പിക്കുകയാണ് (പരമാവധി ടോക്കൺ വിൻഡോയിൽ ഫിറ്റ് ചെയ്യാനാകുന്നതിൽ നിന്നും കൂടുതല് ഉദാഹരണങ്ങൾ ഉപയോഗപ്പെടുത്തുന്നതിനായി) - അങ്ങനെ ഒരു `_custom_` മോഡലിന്റെ പതിപ്പ് വിനിയോഗിക്കുന്നു, ഇത് ഇനി ഇഫിൻറെൻസ് സമയത്ത് ഉദാഹരണങ്ങൾ നൽകേണ്ടതില്ല. ഇതിലൂടെ ഞങ്ങളുടെ പ്രൊംപ്റ്റ് ഡിസൈന്റെ ഫലപ്രാപ്തിയും മെച്ചപ്പെടുന്നു (ടോക്കൺ വിൻഡോ മറ്റ് കാര്യങ്ങൾക്കായി കൂടുതൽ സ്വാതന്ത്ര്യമുണ്ട്) കൂടാതെ ചിലവ് കുറയ്ക്കുന്നതിലും സഹായിക്കുന്നു (ഇഫിൻറെൻസ് സമയത്ത് മോഡലിലേക്കയയ്ക്കേണ്ട ടോക്കണുകളുടെ എണ്ണം കുറയ്ക്കുന്നത് കൊണ്ടും).

ഫൈൻ-ട്യൂണിംഗിന് 4 ഘട്ടങ്ങളുണ്ട്:
1. പരിശീലന ഡാറ്റ തയ്യാറാക്കുകയും അപ്‌ലോഡ് ചെയ്യുകയും ചെയ്യുക.
1. ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ ലഭിക്കുന്നതിന് പരിശീലന ജോലി നടത്തുക.
1. ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ വിലയിരുത്തുക, ഗുണനിലവാരം മെച്ചപ്പെടുത്താൻ ആവർത്തിക്കുക.
1. സന്തോഷമുള്ളപ്പോൾ ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ ഇഫിൻറെൻസിനായി വിനിയോഗിക്കുക.

എല്ലാ ഫൗണ്ടേഷൻ മോഡലുകളും ഫൈൻ-ട്യൂണിംഗ് പിന്തുണയ്ക്കുന്നില്ല - ഏറ്റവും പുതിയ വിവരങ്ങൾക്ക് [OpenAI ഡോക്യുമെന്റേഷൻ](https://platform.openai.com/docs/guides/fine-tuning/what-models-can-be-fine-tuned?WT.mc_id=academic-105485-koreyst) പരിശോധിക്കുക. നിങ്ങൾ മുമ്പ് ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡലും വീണ്ടും ഫൈൻ-ട്യൂൺ ചെയ്യാവുന്നതാണ്. ഈ ട്യൂട്ടോറിയലിൽ, `gpt-35-turbo` എന്ന താരം ഫൗണ്ടേഷൻ മോഡലായികൊണ്ട് ഫൈൻ-ട്യൂണിംഗ് നടത്തുന്നു (മുകളിൽ കൊടുത്ത നോട്ടിനു് സമാനമായി ഇപ്പോൾ പിന്തുണയുള്ള മറ്റൊരു മോഡൽ ഉപയോഗിക്കുക).

---


### ഘട്ടം 1.1: നിങ്ങളുടെ ഡാറ്റാസെറ്റ് തയ്യാറാക്കുക

ഒരു ഘടകത്തെക്കുറിച്ച് ക്വശ്ചനുകൾക്ക് ലിമെരിക് ശൈലി കണക്കിലെടുക്കുന്ന ചോദ്യങ്ങൾക്ക് മറുപടി നൽകുന്നതിലൂടെ നിങ്ങളെ ഘടകങ്ങളുടെ പീരിയഡിക് പട്ടിക മനസിലാക്കാൻ സഹായിക്കുന്ന ഒരു ചാറ്റ്ബോട്ട് നമുക്ക് നിർമ്മിക്കാം. _ഈ_ ലളിതമായ ട്യൂട്ടോറിയലിൽ, ഞങ്ങൾ മോഡലിനെ പരിശീലിപ്പിക്കാൻ ഡാറ്റയുടെ പ്രതീക്ഷിക്കുന്ന രൂപം കാണിക്കുന്ന ചില സാമ്പിൾ മറുപടികളുടെ ഡാറ്റാസെറ്റ് മാത്രമേ സൃഷ്ടിക്കാറുള്ളൂ. യാഥാർത്ഥ്യ ഉപയോഗ സാഹചര്യത്തിൽ, നിങ്ങൾക്ക് കൂടുതൽ ഉദാഹരണങ്ങളടങ്ങിയ ഡാറ്റാസെറ്റ് സൃഷ്ടിക്കേണ്ടി വരും. നിങ്ങളുടെ അപ്ലിക്കേഷൻ ഡൊമെയിനിൽ ഒരു ഓപ്പൺ ഡാറ്റാസെറ്റ് ലഭ്യമെങ്കിൽ അത് ഉപയോഗിച്ചു മടക്കത്തിൽ ഉപയോക്തൃ നിർദ്ദേശങ്ങൾക്ക് ഫൈന്ട്യൂണിംഗിന് അനുസൃതമാക്കാൻ നിങ്ങൾക്ക് കഴിയും.

`gpt-35-turbo`-നെയാണ് ഞങ്ങൾ കേന്ദ്രീകരിക്കുന്നത്, ഒറ്റ ടേൺ മറുപടി (ചാറ്റ് സമ്പൂര്‌ണ്ണം) അന്വേഷിക്കുന്നതിനാൽ OpenAI ചാറ്റ് സമ്പൂര്‌ണ്ണ ആവശ്യകതകൾ പ്രതിബിംബിപ്പിക്കുന്ന [ഈ നിർദ്ദേശിച്ച ഫോർമാറ്റ്](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) ഉപയോഗിച്ച് ഉദാഹരണങ്ങൾ സൃഷ്ടിക്കാം. നിങ്ങൾ ബഹു-ടേൺ സംവാദ ഉള്ളടക്കം പ്രതീക്ഷിക്കുന്നെങ്കിൽ, അതിലെ ഏതെല്ലാം സന്ദേശങ്ങൾ ഫൈന്ട്യൂണിംഗ് പ്രക്രിയയിൽ ഉപയോഗിക്കണം (അല്ലെങ്കിൽ ഉപയോഗിക്കരുത്) എന്ന് സൂചിപ്പിക്കാൻ `weight` പാരാമീറ്റർ ഉൾക്കൊള്ളുന്ന [ബഹു-ടേൺ ഉദാഹരണ ഫോർമാറ്റ്](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) നിങ്ങൾ ഉപയോഗിക്കും.

ഞങ്ങൾ ഇവിടെ ലളിതമായ ഒറ്റ-ടേൺ ഫോർമാറ്റ് ആണ് ഉപയോഗിക്കുന്നത്. ഡാറ്റ [jsonl ഫോർമാറ്റിലാണുള്ളത്](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst), ഓരോ വരിയിലും 1 റെക്കോർഡുണ്ട്, ഓരോന്നും JSON-ഫോർമാറ്റിലുള്ള ഒബ്ജക്റ്റായി പ്രതിനിധാനം ചെയ്യും. താഴെയുള്ള ചെറിയ ഉദാഹരണത്തിൽ 2 റെക്കോർഡുകൾ കാണിക്കുന്നു - പൂർണ്ണ സാമ്പിൾ സെറ്റ് (10 ഉദാഹരണങ്ങൾ) കാണുന്നതിന് [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) കാണുക, നാം ഫൈന്ട്യൂണിംഗ് ട്യൂട്ടോറിയലിനായി ഉപയോഗിക്കും. **കുറിപ്പ്:** ഓരോ റെക്കോർഡും _ഒറ്റ വരിയിലായും_ നിർവചിക്കണം (സാധാരണ ഒരു ഫോർമാറ്റ് ചെയ്ത JSON ഫയലിൽ കാണുന്ന പോലെ വരികളിലായി വിഭജിക്കരുത്)

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

യാഥാർത്ഥ്യ ഉപയോഗത്തിനായി നിങ്ങൾക്ക് നല്ല ഫലങ്ങൾ നേടുന്നതിനായി വളരെ കൂടുതൽ ഉദാഹരണങ്ങളടങ്ങിയ സെറ്റ് ആവശ്യമാകും - മാറ്റം മറുപടികളുടെ ഗുണമേന്മയിലും ഫൈന്ട്യൂണിംഗിന്റെ സമയം/ച്ചെലവുകളിലും ഇടയിലുണ്ടാകും. നാം ഒരു ചെറിയ സെറ്റ് ഉപയോഗിക്കുന്നു അതുകൊണ്ടാണ് ഫൈന്ട്യൂണിംഗ് പ്രക്രിയ കടപ്പാട് സാദ്ധ്യമാക്കാൻ. കൂടുതൽ സങ്കീർണ്ണമായ ഫൈന്ട്യൂണിംഗ് ട്യൂട്ടോറിയലിനായി [ഈ OpenAI കുക്ക്ബുക്ക് ഉദാഹരണം](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) കാണുക.


---

### ഘട്ടം 1.2 നിങ്ങളുടെ ഡാറ്റാസെറ്റ് അപ്‌ലോഡ് ചെയ്യുക

ഡാറ്റ ഫയൽസ് API ഉപയോഗിച്ച് അപ്‌ലോഡ് ചെയ്യുക [ഇവിടെ വിവരിച്ചിരിക്കുന്നു](https://platform.openai.com/docs/guides/fine-tuning/upload-a-training-file). ഈ കോഡ് പ്രവർത്തിപ്പിക്കാൻ, നിങ്ങൾക്ക് ആദ്യം താഴെയുള്ള ഘട്ടങ്ങൾ പൂർത്തിയാക്കിയിരിക്കണം:
 - `openai` പython പാക്കേജ് ഇൻസ്റ്റാൾ ചെയ്തിരിക്കണം (സജ്ജീകരണത്തിനായി >=0.28.0 വേർഷൻ ഉപയോഗിക്കുക)
 - നിങ്ങളുടെ OpenAI API കീയെ `OPENAI_API_KEY` എന്ന എൻവയോൺമെന്റ് വേരിയബിൾ ആയി സെറ്റ് ചെയ്തിരിക്കണം
കൂടുതൽ അറിയാൻ, കോഴ്‌സിനായി നൽകിയ [സജ്ജീകരണ ഗൈഡ്](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst) കാണുക.

ഇപ്പോൾ, ലൊക്കൽ JSONL ഫയലിൽ നിന്നുള്ള അപ്‌ലോഡിനായി ഫയൽ സൃഷ്ടിക്കാൻ കോഡ് റൺ ചെയ്യുക.


In [24]:
from openai import OpenAI
client = OpenAI()

ft_file = client.files.create(
  file=open("./training-data.jsonl", "rb"),
  purpose="fine-tune"
)

print(ft_file)
print("Training File ID: " + ft_file.id)

FileObject(id='file-JdAJcagdOTG6ACNlFWzuzmyV', bytes=4021, created_at=1715566183, filename='training-data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)
Training File ID: file-JdAJcagdOTG6ACNlFWzuzmyV


---

### ഘട്ടം 2.1: SDK ഉപയോഗിച്ച് ഫൈൻ-ട്യൂണിംഗ് ജോലി സൃഷ്ടിക്കുക


In [25]:
from openai import OpenAI
client = OpenAI()

ft_filejob = client.fine_tuning.jobs.create(
  training_file=ft_file.id, 
  model="gpt-3.5-turbo"
)

print(ft_filejob)
print("Fine-tuning Job ID: " + ft_filejob.id)

FineTuningJob(id='ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', created_at=1715566184, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-EZ6ag0n0S6Zm8eV9BSWKmE6l', result_files=[], seed=830529052, status='validating_files', trained_tokens=None, training_file='file-JdAJcagdOTG6ACNlFWzuzmyV', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)
Fine-tuning Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh


---

### ഘട്ടം 2.2: ജോബിന്റെ കുടുംബസ്ഥിതി പരിശോധിക്കുക

`client.fine_tuning.jobs` API ഉപയോഗിച്ച് നിങ്ങൾ ചെയ്യാൻ കഴിയുന്ന ചില കാര്യങ്ങൾ ഇവയാണ്:
- `client.fine_tuning.jobs.list(limit=<n>)` - അവസാന n ഫൈൻ-ട്യൂണിംഗ് ജോലികളെ ലിസ്റ്റ് ചെയ്യുക
- `client.fine_tuning.jobs.retrieve(<job_id>)` - ഒരു പ്രത്യേക ഫൈൻ-ട്യൂണിംഗ് ജോബിന്റെ വിശദാംശങ്ങൾ നേടുക
- `client.fine_tuning.jobs.cancel(<job_id>)` - ഒരു ഫൈൻ-ട്യൂണിംഗ് ജോബ് റദ്ദാക്കുക
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<b>)` - ജോലിയിൽ നിന്ന് n വരെ ഇവന്റ്‌സ് ലിസ്റ്റ് ചെയ്യുക
- `client.fine_tuning.jobs.create(model="gpt-35-turbo", training_file="your-training-file.jsonl", ...)`

പ്രക്രിയയുടെ ആദ്യഘട്ടം ഡാറ്റ ശരിയായ ഫോർമാറ്റിലാണെന്ന് ഉറപ്പാക്കാൻ _ട്രെയിനിംഗ് ഫയൽാ സാധുത പരിശോധിക്കുക_ എന്നതാണ്.


In [26]:
from openai import OpenAI
client = OpenAI()

# List 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(ft_filejob.id)

# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_filejob.id, limit=10)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-GkWiDgZmOsuv4q5cSTEGscY6', created_at=1715566184, level='info', message='Validating training file: file-JdAJcagdOTG6ACNlFWzuzmyV', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-3899xdVTO3LN7Q7LkKLMJUnb', created_at=1715566184, level='info', message='Created fine-tuning job: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=False)

In [30]:
# Once the training data is validated
# Track the job status to see if it is running and when it is complete
from openai import OpenAI
client = OpenAI()

response = client.fine_tuning.jobs.retrieve(ft_filejob.id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh
Status: running
Trained Tokens: None


---

### ഘട്ടം 2.3: പുരോഗതി നിരീക്ഷിക്കാൻ ഇവന്റുകൾ ട്രാക്ക് ചെയ്യുക


In [44]:
# You can also track progress in a more granular way by checking for events
# Refresh this code till you get the `The job has successfully completed` message
response = client.fine_tuning.jobs.list_events(ft_filejob.id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 85/100: training loss=0.14
Step 86/100: training loss=0.00
Step 87/100: training loss=0.00
Step 88/100: training loss=0.07
Step 89/100: training loss=0.00
Step 90/100: training loss=0.00
Step 91/100: training loss=0.00
Step 92/100: training loss=0.00
Step 93/100: training loss=0.00
Step 94/100: training loss=0.00
Step 95/100: training loss=0.08
Step 96/100: training loss=0.05
Step 97/100: training loss=0.00
Step 98/100: training loss=0.00
Step 99/100: training loss=0.00
Step 100/100: training loss=0.00
Checkpoint created at step 80 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyyF2:ckpt-step-80
Checkpoint created at step 90 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyzhK:ckpt-step-90
New fine-tuned model created: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz
The job has successfully completed


### ഘട്ടം 2.4: OpenAI ഡാഷ്ബോർഡിൽ നില അറിയുക


നിങ്ങൾ ഓപ്പൺ‌എഐ വെബ്സൈറ്റ് സന്ദർശിച്ച് പ്ലാറ്റ്ഫോമിന്റെ _Fine-tuning_ വിഭാഗം പരിശോധിച്ചുകൊണ്ടും സ്റ്റാറ്റസ് കാണാം. ഇതോടെ നിങ്ങൾക്ക് നിലവിലെ ജോബ് സ്റ്റാറ്റസ് കാണാനും മുൻ ജോബ് നടപ്പാക്കലുകളുടെ ചരിത്രം പിന്തുടരാനും കഴിയും. ഈ സ്ക്രീൻഷോട്ടിൽ, മുൻനിശ്ചയ പ്രവർത്തനം പരാജയപ്പെട്ടതും രണ്ടാമത്തെ പ്രവർത്തനം വിജയിച്ചതും നിങ്ങൾക്ക് കാണാം. പശ്ചാത്തലത്തിന്, ആദ്യ പ്രവർത്തനം തെറ്റായ ഫോർമാറ്റിലുള്ള JSON ഫയൽ ഉപയോഗിച്ചതിനെ തുടർന്ന് ഇത് സംഭവിച്ചത് ആണ് - അത് ശരിയാക്കിയപ്പോൾ രണ്ടാമത്തെ പ്രവർത്തനം വിജയകരമായി പൂർത്തിയാക്കി മോഡൽ ഉപയോഗത്തേക്കായി ലഭ്യമായുപോയി.

![Fine-tuning job status](../../../../../translated_images/ml/fine-tuned-model-status.563271727bf7bfba.webp)


താഴെ കാണിച്ചിരിക്കുന്നതുപോലെ, വിഷ്വൽ ഡാഷ്ബോർഡിൽ താഴേക്ക് സ്ക്രോൾ ചെയ്‌താൽ നിങ്ങൾക്ക് സ്റ്റാറ്റസ് സന്ദേശങ്ങളും മീറ്റ്രിക്കുകളും കാണാൻ കഴിയും:

| സന്ദേശങ്ങൾ | മീറ്റ്രിക്കുകൾ |
|:---|:---|
| ![Messages](../../../../../translated_images/ml/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/ml/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### ഘടകം 3.1: ഐഡി പിടിക്കുക & കോഡിൽ ഫൈൻ-ട്യൂണഡ് മോഡൽ പരിശോധിക്കുക


In [46]:
# Retrieve the identity of the fine-tuned model once ready
response = client.fine_tuning.jobs.retrieve(ft_filejob.id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)

Fine-tuned Model ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz


In [ ]:
# You can then use that model to generate completions from the SDK as shown
# Or you can load that model into the OpenAI Playground (in the UI) to validate it from there.
from openai import OpenAI
client = OpenAI()

completion = client.responses.create(
  model=fine_tuned_model_id,
  input=[
    {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Strontium"},
  ],
  store=False,
)
print(completion.output_text)


ChatCompletionMessage(content="Strontium, a metal so bright - It's in fireworks, a dazzling sight - It's in bones, you see - And in tea, it's the key - It's the fortieth, so pure, that's the right", role='assistant', function_call=None, tool_calls=None)


---

### ഘടകം 3.2: ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ പ്ലേഗ്രൗണ്ടിൽ ലോഡ് ചെയ്ത് പരിശോധിക്കുക

നിങ്ങൾക്ക് ഇപ്പോൾ ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ രണ്ട് മാർഗങ്ങളിൽ പരിശോധന നടത്താം. ആദ്യം, നിങ്ങൾക്ക് പ്ലേഗ്രൗണ്ട് സന്ദർശിച്ച് മോഡലുകൾ ഡ്രോപ്പ്-ഡൗൺ ഉപയോഗിച്ച് പുതിയ ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ തിരഞ്ഞെടുക്കാം. മറ്റൊന്ന്, ഫൈൻ-ട്യൂണിംഗ് പാനലിൽ കാണിക്കുന്ന "Playground" തെരഞ്ഞെടുപ്പ് (മുകളിൽ സ്ക്രിൻഷോട്ട് കാണുക) ഉപയോഗിച്ച് തുടർന്നുള്ള _comaritive_ കാഴ്ച തുറക്കാം, ഇത് ഫൗണ്ടേഷൻ മോഡലും ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡലും ഒരു പന്തിൽ തമ്മിൽവെച്ച് വേഗത്തില്‍ വിലയിരുത്തുന്നതിനുള്ളത് ആണ്.

![Fine-tuning job status](../../../../../translated_images/ml/fine-tuned-playground-compare.56e06f0ad8922016.webp)

പരിശീലന ഡേറ്റയിൽ ഉപയോഗിച്ചതുപോലെ സിസ്റ്റം കോൺടെക്സ്റ്റ് ടൈപ്പ് ചെയ്ത് നിങ്ങളുടെ ടെസ്റ്റ് ചോദ്യവും നൽകുക. ഇരുവശവും അതേ കോൺടെക്സ്റ്റ്, ചോദ്യവും അപ്ഡേറ്റ് ആകുന്നതാണ് കാണാൻ. താരതമ്യം നടത്തുക, പുറത്തുവന്ന ഫലങ്ങളിലെ വ്യത്യാസം നിങ്ങൾ കാണും. _ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ നിങ്ങളുടെ ഉദാഹരണങ്ങളിൽ നൽകിയ ഫോർമാറ്റിൽ പ്രതികരണം നൽകുന്നത് ശ്രദ്ധിക്കുക, എന്നാൽ ഫൗണ്ടേഷൻ മോഡൽ സിസ്റ്റം പ്രോമ്പ്റ്റ് പിന്തുടരെയ്ക്കുന്നു_.

![Fine-tuning job status](../../../../../translated_images/ml/fine-tuned-playground-launch.5a26495c983c6350.webp)

താരതമ്യം ഓരോ മോഡലിന്റെയും ടോക്കൺ എണ്ണം, ഇൻഫെറൻസ് വേണ്ടി എടുത്ത സമയം എന്നിവ നൽകുന്നതും കാണും. **ഈ ഉദാഹരണം ഒരു ലളിതമായത് മാത്രമാണ്, പ്രക്രിയ കാണിക്കാൻ, യഥാർത്ഥ ഡേറ്റാ സെറ്റോ സന്നിവേശമോ കാണിക്കുന്നതല്ല**. രണ്ടു സാമ്പിളുകളും ടോക്കണുകളുടെ എണ്ണം ഏകമാണ് (സിസ്റ്റം കോൺടെക്സ്റ്റും ഉപയോക്തൃ പ്രോമ്പ്റ്റും ഒരുപോലെയാണ്), ഫൈൻ-ട്യൂൺ ചെയ്ത മോഡൽ ഇടപാടിൽ കൂടുതൽ സമയം എടുക്കുന്നു (ആദ്യകം മോഡൽ).

യഥാർത്ഥ സാഹചര്യങ്ങളിൽ നിങ്ങൾ ഇത്തരമൊരു ടോയി ഉദാഹരണം ഉപയോഗിക്കുകയില്ല, പക്ഷെ യഥാർത്ഥ ഡേറ്റയുമായി (ഉദാഹരണത്തിന്, ഉപഭോക്തൃ സേവനത്തിന് പ്രൊഡക്റ്റ് കാറ്റലോഗ്) ഫൈൻ-ട്യൂണിംഗ് ചെയ്‌താൽ പ്രതികരണത്തിന്റെ ഗുണമേന്മ കൂടുതൽ തെളിയിക്കും. _അത്_ സാഹചര്യം, ഫൗണ്ടേഷൻ മോഡലിൽ സമവായ_Response ഗുണമേന്മ ലഭിക്കാൻ കൂടുതൽ കസ്റ്റം പ്രോമ്പ്റ്റ് എഞ്ചിനീയറിംഗ് ആവശ്യമായിരിക്കും, ഇത് ടോക്കൺ ഉപയോഗവും ഇൻഫെറൻസിനുള്ള പ്രോസസ്സിംഗ് സമയവും വർധിപ്പിക്കും. _ഇതിന് ശ്രമിക്കാൻ, ഫൈൻ-ട്യൂണിംഗ് ഉദാഹരണങ്ങൾ OpenAI കുക്ക്ബുക്കിൽ കാണുക._

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
